In [23]:
import torch
import plotly.graph_objects as go

from src.gp_ccm import run_sigGPCCM_experiment
from src.sp_ccm import run_ccm_experiment

torch.set_printoptions(sci_mode = False)

In [24]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

Using device: cuda



# Generate data

In [84]:
co2_norm = torch.load("data/CO2_vostok_stan_400kyr_timeseries.pt").to(torch.float32)

# Initalise
g = torch.tensor([0.2, 0.1, 0.1])

true_offset = -2

for t in range(2, co2_norm.shape[0] + 1):
    
    # Co2 is external forcing 
    g_next = g_next = (g[t] * (1.4 - (1.9 * g[t]) - (0.3 * co2_norm[t + true_offset])))

    g = torch.concat((g, g_next.unsqueeze(0)))

g_norm = g.sub(g.mean(dim = -1).unsqueeze(-1)).div(g.std(dim = -1).unsqueeze(-1))[0:401]

In [117]:
MAX = 401

fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, co2_norm.shape[0])[0:MAX], y = co2_norm[0:MAX],
                    mode = 'lines',
                    name = 'F',
                    line_color = "red"))

fig.add_trace(go.Scatter(x = torch.arange(0, co2_norm.shape[0])[0:MAX], y = g_norm[0:MAX],
                    mode = 'lines',
                    name = 'G',
                    line_color = "purple"))

fig.update_layout(title = 'Synchronous time series',
                   xaxis_title = 't',
                   yaxis_title = 'values')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")
fig.update_layout(xaxis_range=[-2, MAX])

fig.update_layout(autosize = False, width = 1000, height = 400)

fig.show()

In [120]:
# GLOBALS
k = 3
N_TRAIN = torch.tensor([100]).to(device)

##############
### GP-CCM ###
##############

sig_filter = torch.ones(size = (k, )).to(device)
sig_shift = torch.tensor(sig_filter.shape[0] - 1).to(device) + 1 # k -1 

# both methods rely on noise for numerical stability
NOISE_SCALE = torch.tensor([0.05], device = device) # for diagonal
NOISE_SCALE_low = torch.tensor([0.01], device = device) # for diagonal
NOISE_SCALE_medium = torch.tensor([0.02], device = device) # for diagonal

RBF_SCALE = torch.tensor([0.4], device = device) # 25 before

############
### ECCM ###
############

ccm_filter = torch.ones(size = (k, )).to(device) # same as sig filter
ccm_shift = torch.tensor(ccm_filter.shape[0] - 1).to(device) + 1

# CO2 -> G

In [87]:
### sig-GP_CCM ###
CO2G_gpccm_rho_mean, CO2G_gpccm_rho_sd, CO2G_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
    causal_x = co2_norm.to(device),
    causal_y = g_norm.to(device),
    sig_filter = sig_filter.to(device),
    sig_shift = sig_shift.to(device),
    rbf_scale = RBF_SCALE, 
    noise_scale = NOISE_SCALE, 
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean 0.503
Rho std 0.125
Rho indep. p95 0.291


In [ ]:
### CCM ###
CO2G_ccm_rho_mean, CO2G_ccm_rho_sd, CO2G_ccm_rho_ind_p95, CO2G_ccm_noise =  run_ccm_experiment(
    causal_x = co2_norm.to(device),
    causal_y = g_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

In [99]:
def vis_synchrony(shift_results_gpccm, k, shifts, true_shift, isGPCCM, legend_y_displacement = 1.01):

    if (isGPCCM == True):
        line_col = "#C00000"
        bounds_line_col = 'rgba(243, 176, 210, 0.8)'
        bounds_fill_col = 'rgba(243, 176, 210, 0.3)'

    else:
        line_col = "rgba(103, 95, 114, 1.0)"
        bounds_line_col = "rgba(134, 122, 151, 0.8)"
        bounds_fill_col = "rgba(134, 122, 151, 0.3)"

    
    # Initalise
    fig = go.Figure()

    # left to right
    fig.add_vrect(x0 = -8, x1 = -(k - 1), line_width = 0, fillcolor = "green", opacity = 0.1)
    fig.add_vrect(x0 = -(k - 1), x1 = 0.0, line_width = 0, fillcolor = "grey", opacity = 0.2)
    fig.add_vrect(x0 = 0., x1 = 8., line_width = 0, fillcolor = "red", opacity = 0.1)

    # separators
    fig.add_vline(x = -(k - 1), line_width = 0.5, line_color = "black", opacity = 0.5)
    fig.add_vline(x = 0.0, line_width = 0.5, line_color = "black", opacity = 0.5)

    # fig.add_vline(x = true_shift - 1, line_width = 1.0, line_color = "green", line_dash = "dash", opacity = 0.8)

    fig.add_trace(go.Scatter(x = -shifts, y = shift_results_gpccm[:, 2], # reversing the meaning of x
                        mode = 'lines',
                        name = 'rho ind. p95',
                        line_color = "black",
                        line_dash = "dot",
                        ))

    fig.add_trace(go.Scatter(x = -shifts, y = shift_results_gpccm[:, 0], # reversing the meaning of x
                        mode = 'lines+markers',
                        name = 'mean rho +/- 1 sd',
                        line_color = line_col))


    fig.add_trace(go.Scatter(
        name = "",
        x = - shifts,
        y = shift_results_gpccm[:, 0] + shift_results_gpccm[:, 1].mul(1),
        marker = dict(color = bounds_line_col),
        showlegend = False,
        mode = 'lines'
    ))

    fig.add_trace(go.Scatter(
        name = "",
        x = - shifts,
        y = shift_results_gpccm[:, 0] - shift_results_gpccm[:, 1].mul(1),
        marker = dict(color = bounds_line_col),
        showlegend = False,
        mode = 'lines',
        fillcolor = bounds_fill_col,
        fill = 'tonexty'
    ))

    # empty
    fig.update_layout(title = '',
                    xaxis_title = '',
                    yaxis_title = '')

    fig.update_layout(template = "simple_white")
    fig.update_layout(font_family = "Lato")
    fig.update_layout(width = 600, height = 350)
    fig.update_layout(xaxis_range = [-8.0, 8.0])
    fig.update_layout(yaxis_range = [-0.1, 1.01])

    fig.update_layout(legend = dict(x = 0, y = legend_y_displacement, bgcolor = "rgba(0,0,0,0)"))

    fig.show()

# Loop for synchrony

In [121]:
# 2 min to run
shifts = torch.arange(-8, 8 + 1, 1)

shift_results_gpccm_CO2G = torch.zeros(size = (shifts.shape[0], 3))

for i, s in enumerate(shifts):
    print("Shift", s.item())
    CO2G_gpccm_rho_mean, CO2G_gpccm_rho_sd, CO2G_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
        causal_x = co2_norm.to(device),
        causal_y = g_norm.to(device),
        sig_filter = sig_filter.to(device),
        sig_shift = s.to(device), # try a different shift
        rbf_scale = RBF_SCALE, 
        noise_scale = NOISE_SCALE, 
        n_train = N_TRAIN.to(device),
        device = device)

    shift_results_gpccm_CO2G[i, 0] = CO2G_gpccm_rho_mean
    shift_results_gpccm_CO2G[i, 1] = CO2G_gpccm_rho_sd
    shift_results_gpccm_CO2G[i, 2] = CO2G_gpccm_rho_ind_p95

    print(" ")

vis_synchrony(shift_results_gpccm_CO2G, k, shifts, true_offset, True)

Shift -8
Rho mean 0.198
Rho std 0.08
Rho indep. p95 0.242
 
Shift -7
Rho mean 0.203
Rho std 0.076
Rho indep. p95 0.235
 
Shift -6
Rho mean 0.212
Rho std 0.08
Rho indep. p95 0.253
 
Shift -5
Rho mean 0.216
Rho std 0.084
Rho indep. p95 0.241
 
Shift -4
Rho mean 0.227
Rho std 0.083
Rho indep. p95 0.254
 
Shift -3
Rho mean 0.234
Rho std 0.08
Rho indep. p95 0.223
 
Shift -2
Rho mean 0.251
Rho std 0.079
Rho indep. p95 0.237
 
Shift -1
Rho mean 0.269
Rho std 0.082
Rho indep. p95 0.236
 
Shift 0
Rho mean 0.304
Rho std 0.09
Rho indep. p95 0.24
 
Shift 1
Rho mean 0.346
Rho std 0.096
Rho indep. p95 0.242
 
Shift 2
Rho mean 0.38
Rho std 0.103
Rho indep. p95 0.25
 
Shift 3
Rho mean 0.416
Rho std 0.111
Rho indep. p95 0.233
 
Shift 4
Rho mean 0.391
Rho std 0.101
Rho indep. p95 0.26
 
Shift 5
Rho mean 0.358
Rho std 0.098
Rho indep. p95 0.223
 
Shift 6
Rho mean 0.305
Rho std 0.09
Rho indep. p95 0.234
 
Shift 7
Rho mean 0.277
Rho std 0.095
Rho indep. p95 0.245
 
Shift 8
Rho mean 0.249
Rho std 0.102
Rho 

In [91]:
vis_synchrony(shift_results_gpccm_CO2G, k, shifts, true_offset, True)

In [92]:
# 7 min to run
shifts = torch.arange(-8, 8 + 1, 1)

shift_results_ccm_CO2G = torch.zeros(size = (shifts.shape[0], 3))

for i, s in enumerate(shifts):
    print("Shift", s.item())

    CO2G_ccm_rho_mean, CO2G_ccm_rho_sd, CO2G_ccm_rho_ind_p95, CO2G_ccm_noise =  run_ccm_experiment(
        causal_x = co2_norm.to(device),
        causal_y = g_norm.to(device),
        ccm_filter = ccm_filter.to(device),
        ccm_shift = s.to(device), # try a different shift
        n_train = N_TRAIN.to(device),
        device = device)

    shift_results_ccm_CO2G[i, 0] = CO2G_ccm_rho_mean
    shift_results_ccm_CO2G[i, 1] = CO2G_ccm_rho_sd
    shift_results_ccm_CO2G[i, 2] = CO2G_ccm_rho_ind_p95

    print(" ")

Shift -8
Rho mean 0.316
Rho std 0.101
Rho indep. p95 0.579
Added noise 0.0
 
Shift -7
Rho mean 0.372
Rho std 0.099
Rho indep. p95 0.578
Added noise 0.0
 
Shift -6
Rho mean 0.424
Rho std 0.091
Rho indep. p95 0.591
Added noise 0.0
 
Shift -5
Rho mean 0.474
Rho std 0.083
Rho indep. p95 0.588
Added noise 0.0
 
Shift -4
Rho mean 0.523
Rho std 0.075
Rho indep. p95 0.602
Added noise 0.0
 
Shift -3
Rho mean 0.577
Rho std 0.059
Rho indep. p95 0.589
Added noise 0.0
 
Shift -2
Rho mean 0.639
Rho std 0.048
Rho indep. p95 0.577
Added noise 0.0
 
Shift -1
Rho mean 0.706
Rho std 0.04
Rho indep. p95 0.562
Added noise 0.0
 
Shift 0
Rho mean 0.776
Rho std 0.033
Rho indep. p95 0.585
Added noise 0.0
 
Shift 1
Rho mean 0.848
Rho std 0.027
Rho indep. p95 0.58
Added noise 0.0
 
Shift 2
Rho mean 0.909
Rho std 0.023
Rho indep. p95 0.577
Added noise 0.0
 
Shift 3
Rho mean 0.951
Rho std 0.022
Rho indep. p95 0.607
Added noise 0.0
 
Shift 4
Rho mean 0.959
Rho std 0.02
Rho indep. p95 0.569
Added noise 0.0
 
Shift 5

In [98]:
# Visualise ccm 
vis_synchrony(shift_results_ccm_CO2G, k, shifts, true_offset, False, legend_y_displacement = 0.01)

# G -> CO2

In [30]:
### sig-GP_CCM ###
GCO2_gpccm_rho_mean, GCO2_gpccm_rho_sd, GCO2_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
    causal_x = g_norm.to(device),
    causal_y = co2_norm.to(device),
    sig_filter = sig_filter.to(device),
    sig_shift = sig_shift.to(device),
    rbf_scale = RBF_SCALE, 
    noise_scale = NOISE_SCALE, 
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean 0.217
Rho std 0.056
Rho indep. p95 0.258


In [31]:
### CCM ###
GCO2_ccm_rho_mean, GCO2_ccm_rho_sd, GCO2_ccm_rho_ind_p95, GCO2_ccm_noise =  run_ccm_experiment(
    causal_x = g_norm.to(device),
    causal_y = co2_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

We get nan's and have to increase the noise level.
Rho mean 0.567
Rho std 0.137
Rho indep. p95 0.504
Added noise 0.025


In [114]:
# 2 min to run
# GPCCM loop
shifts = torch.arange(-8, 8 + 1, 1)

shift_results_gpccm_GCO2 = torch.zeros(size = (shifts.shape[0], 3))

for i, s in enumerate(shifts):
    print("Shift", s.item())
    GCO2_gpccm_rho_mean, GCO2_gpccm_rho_sd, GCO2_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
        causal_x = g_norm.to(device),
        causal_y = co2_norm.to(device),
        sig_filter = sig_filter.to(device),
        sig_shift = s.to(device), # try a different shift
        rbf_scale = RBF_SCALE, 
        noise_scale = NOISE_SCALE_low, # stable 
        n_train = N_TRAIN.to(device),
        device = device)

    shift_results_gpccm_GCO2[i, 0] = GCO2_gpccm_rho_mean
    shift_results_gpccm_GCO2[i, 1] = GCO2_gpccm_rho_sd
    shift_results_gpccm_GCO2[i, 2] = GCO2_gpccm_rho_ind_p95

    print(" ")

vis_synchrony(shift_results_gpccm_GCO2, k, shifts, true_offset, True)

Shift -8
Rho mean 0.107
Rho std 0.146
Rho indep. p95 0.192
 
Shift -7
Rho mean 0.123
Rho std 0.149
Rho indep. p95 0.19
 
Shift -6
Rho mean 0.143
Rho std 0.162
Rho indep. p95 0.178
 
Shift -5
Rho mean 0.15
Rho std 0.175
Rho indep. p95 0.191
 
Shift -4
Rho mean 0.175
Rho std 0.151
Rho indep. p95 0.175
 
Shift -3
Rho mean 0.171
Rho std 0.148
Rho indep. p95 0.185
 
Shift -2
Rho mean 0.177
Rho std 0.136
Rho indep. p95 0.174
 
Shift -1
Rho mean 0.174
Rho std 0.12
Rho indep. p95 0.185
 
Shift 0
Rho mean 0.152
Rho std 0.104
Rho indep. p95 0.183
 
Shift 1
Rho mean 0.134
Rho std 0.089
Rho indep. p95 0.18
 
Shift 2
Rho mean 0.114
Rho std 0.078
Rho indep. p95 0.191
 
Shift 3
Rho mean 0.099
Rho std 0.07
Rho indep. p95 0.179
 
Shift 4
Rho mean 0.082
Rho std 0.068
Rho indep. p95 0.171
 
Shift 5
Rho mean 0.072
Rho std 0.068
Rho indep. p95 0.176
 
Shift 6
Rho mean 0.061
Rho std 0.07
Rho indep. p95 0.172
 
Shift 7
Rho mean 0.048
Rho std 0.067
Rho indep. p95 0.182
 
Shift 8
Rho mean 0.031
Rho std 0.056
R

In [102]:
# 15 min to run
# CCM loop
shifts = torch.arange(-8, 8 + 1, 1)

shift_results_ccm_GCO2 = torch.zeros(size = (shifts.shape[0], 3))

for i, s in enumerate(shifts):
    print("Shift", s.item())

    GCO2_ccm_rho_mean, GCO2_ccm_rho_sd, GCO2_ccm_rho_ind_p95, GCO2_ccm_noise =  run_ccm_experiment(
        causal_x = g_norm.to(device),
        causal_y = co2_norm.to(device),
        ccm_filter = ccm_filter.to(device),
        ccm_shift = s.to(device), # try a different shift
        n_train = N_TRAIN.to(device),
        device = device)

    shift_results_ccm_GCO2[i, 0] = GCO2_ccm_rho_mean
    shift_results_ccm_GCO2[i, 1] = GCO2_ccm_rho_sd
    shift_results_ccm_GCO2[i, 2] = GCO2_ccm_rho_ind_p95

    print(" ")

vis_synchrony(shift_results_ccm_GCO2, k, shifts, true_offset, False)

Shift -8
We get nan's and have to increase the noise level.
Rho mean 0.811
Rho std 0.02
Rho indep. p95 0.539
Added noise 0.025
 
Shift -7
We get nan's and have to increase the noise level.
Rho mean 0.867
Rho std 0.014
Rho indep. p95 0.496
Added noise 0.025
 
Shift -6
We get nan's and have to increase the noise level.
Rho mean 0.915
Rho std 0.008
Rho indep. p95 0.484
Added noise 0.025
 
Shift -5
We get nan's and have to increase the noise level.
Rho mean 0.948
Rho std 0.004
Rho indep. p95 0.521
Added noise 0.025
 
Shift -4
We get nan's and have to increase the noise level.
Rho mean 0.971
Rho std 0.002
Rho indep. p95 0.51
Added noise 0.025
 
Shift -3
We get nan's and have to increase the noise level.
Rho mean 0.977
Rho std 0.003
Rho indep. p95 0.531
Added noise 0.025
 
Shift -2
We get nan's and have to increase the noise level.
Rho mean 0.967
Rho std 0.004
Rho indep. p95 0.522
Added noise 0.025
 
Shift -1
We get nan's and have to increase the noise level.
Rho mean 0.943
Rho std 0.007
Rho

In [106]:
# CCM
torch.save(shift_results_ccm_CO2G, "results/synchrony/shift_results_ccm_CO2G.pt")
torch.save(shift_results_ccm_GCO2, "results/synchrony/shift_results_ccm_GCO2.pt")

# GP-CCM
torch.save(shift_results_gpccm_CO2G, "results/synchrony/shift_results_gpccm_CO2G.pt")
torch.save(shift_results_gpccm_GCO2, "results/synchrony/shift_results_gpccm_GCO2.pt")